<a href="https://colab.research.google.com/github/ginghilterra-collab/Voxtral-TTS-Colab/blob/main/voxtral_tts_t4_optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Voxtral-4B-TTS-2603 T4 GPU Optimized

**🔧 T4 GPU Optimized Version** - Fixed all known issues for Colab T4

## ✅ What's Fixed:
- ✅ cuFFT factory conflicts
- ✅ BFloat16 compatibility for T4
- ✅ Memory optimization for 16GB VRAM
- ✅ CUDA initialization issues
- ✅ Model loading format problems

## 🎯 Optimized Settings:
- **Context Length**: 128 tokens (conservative)
- **GPU Memory**: 35% utilization (5.6GB)
- **Data Type**: Float16 (T4 compatible)
- **Compilation**: Disabled for stability

## Step 1: Verify T4 GPU Setup

First, confirm we have the right GPU and clean up any existing processes:

In [ ]:
# Check GPU and clean environment
import torch
import subprocess
import os
import gc

print("🔍 Checking GPU Setup...")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_cap = f"{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}"
    
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ Memory: {gpu_memory:.1f} GB")
    print(f"✅ Compute Capability: {compute_cap}")
    
    if "T4" in gpu_name:
        print("🎯 Perfect! T4 GPU detected")
    else:
        print("⚠️ Warning: Not T4 GPU, settings may need adjustment")
        
# Clean up everything
print("\n🧹 Cleaning environment...")

# Kill all Python processes
try:
    subprocess.run(['pkill', '-f', 'python'], capture_output=True)
    subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
    print("✅ Killed existing processes")
except:
    pass

# Clear CUDA completely
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()
    print("✅ CUDA memory cleared")

# Fix CUDA environment variables
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print("✅ Environment optimized for T4")

## Step 2: Install Dependencies (T4 Optimized)

Install vLLM and vLLM-Omni with T4-specific configurations:

In [ ]:
# Install system dependencies
!sudo apt-get update -qq
!sudo apt-get install -y build-essential cmake

# Install PyTorch with CUDA 11.8 (T4 compatible)
!pip install --no-cache-dir torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

# Install vLLM (T4 optimized version)
!pip install --no-cache-dir vllm==0.18.1

# Install vLLM-Omni from source
!pip install --no-cache-dir git+https://github.com/vllm-project/vllm-omni.git

# Install additional dependencies
!pip install --no-cache-dir soundfile librosa httpx gradio

print("✅ Dependencies installed for T4 GPU")

## Step 3: Fix BFloat16 for T4 GPU

T4 GPU doesn't support BFloat16. We'll patch the source code:

In [ ]:
import os
import glob
import re

print("🔧 Fixing BFloat16 compatibility for T4 GPU...")

# Find all possible locations of the tokenizer file
search_patterns = [
    '/usr/local/lib/python*/site-packages/vllm_omni/**/voxtral_tts_audio_tokenizer.py',
    '/usr/local/lib/python*/dist-packages/vllm_omni/**/voxtral_tts_audio_tokenizer.py',
    '/usr/local/lib/python*/site-packages/**/voxtral_tts_audio_tokenizer.py',
    '/usr/local/lib/python*/dist-packages/**/voxtral_tts_audio_tokenizer.py'
]

found_files = []
for pattern in search_patterns:
    matches = glob.glob(pattern, recursive=True)
    found_files.extend(matches)

if found_files:
    print(f"📁 Found {len(found_files)} tokenizer file(s)")
    
    for tokenizer_file in found_files:
        print(f"🔧 Processing: {tokenizer_file}")
        
        try:
            # Read the file
            with open(tokenizer_file, 'r') as f:
                content = f.read()
            
            # Multiple BFloat16 patterns to fix
            patterns = [
                (r'dtype=torch\.bfloat16', 'dtype=torch.float16'),
                (r'dtype=torch\.bfloat16,', 'dtype=torch.float16,'),
                (r'dtype=torch\.bfloat16\)', 'dtype=torch.float16)'),
                (r'dtype=torch\.bfloat16  ', 'dtype=torch.float16  '),
                (r'bfloat16', 'float16')  # Catch-all
            ]
            
            changes_made = 0
            for pattern, replacement in patterns:
                if re.search(pattern, content):
                    content = re.sub(pattern, replacement, content)
                    changes_made += 1
            
            if changes_made > 0:
                # Write back the fixed file
                with open(tokenizer_file, 'w') as f:
                    f.write(content)
                print(f"  ✅ Fixed {changes_made} BFloat16 patterns")
            else:
                print(f"  ⚠️ No BFloat16 patterns found (already fixed)")
                
        except Exception as e:
            print(f"  ❌ Error fixing {tokenizer_file}: {e}")
            
else:
    print("⚠️ No tokenizer files found - may not need fixing in newer versions")

# Also search for any other BFloat16 references in vllm_omni
print("\n🔍 Searching for additional BFloat16 references...")
vllm_omni_paths = glob.glob('/usr/local/lib/python*/site-packages/vllm_omni/**/*.py', recursive=True)

bfloat16_files = []
for py_file in vllm_omni_paths[:50]:  # Limit search to first 50 files
    try:
        with open(py_file, 'r') as f:
            content = f.read()
        if 'bfloat16' in content and 'voxtral' in py_file.lower():
            bfloat16_files.append(py_file)
    except:
        continue

if bfloat16_files:
    print(f"📝 Found {len(bfloat16_files)} voxtral files with BFloat16:")
    for file in bfloat16_files[:5]:  # Show first 5
        print(f"  - {file}")
else:
    print("✅ No additional BFloat16 references found")

print("\n✅ BFloat16 compatibility fix completed!")

## Step 4: Start Voxtral Server (T4 Optimized)

Start the server with ultra-conservative T4 settings:

In [ ]:
import subprocess
import time
import os
import signal

print("🚀 Starting Voxtral Server - T4 Optimized Configuration")
print("⏱️ This will take 5-10 minutes to download and load the model...")

# Ultra-conservative T4 settings
server_command = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'mistralai/Voxtral-4B-TTS-2603',
    '--omni',  # TTS functionality
    '--dtype', 'float16',  # T4 compatible
    '--max-model-len', '128',  # Very conservative - was 2048
    '--gpu-memory-utilization', '0.35',  # Very conservative - was 0.7
    '--enforce-eager',  # Disable compilation
    '--config-format', 'mistral',  # Mistral format
    '--load-format', 'mistral',  # Mistral loading
    '--host', '0.0.0.0',
    '--port', '8000',
    '--block-size', '16',  # Smaller blocks for memory
    '--swap-space', '4',  # Use disk swap if needed
    '--disable-log-requests'  # Reduce log spam
]

print(f"\n📋 Server Configuration:")
print(f"  Context Length: 128 tokens")
print(f"  GPU Memory: 35% (~5.6GB)")
print(f"  Data Type: Float16")
print(f"  Compilation: Disabled")

# Create a clean environment
env = os.environ.copy()
env.update({
    'CUDA_LAUNCH_BLOCKING': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:128',
    'PYTHONPATH': '/usr/local/lib/python3.10/site-packages:' + env.get('PYTHONPATH', '')
})

# Start server with the optimized environment
server_process = subprocess.Popen(
    server_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    env=env,
    preexec_fn=os.setsid  # Create new process group
)

print("\n⏳ Server startup initiated...")
print("📊 Monitoring startup progress...")

# Monitor startup for 2 minutes with progress updates
startup_lines = []
for i in range(120):  # 2 minutes
    if server_process.poll() is not None:
        print(f"❌ Server exited early with code: {server_process.returncode}")
        break
        
    # Read any available output
    try:
        line = server_process.stdout.readline()
        if line:
            startup_lines.append(line.strip())
            if i % 10 == 0:  # Show every 10th line
                print(f"📝 {line.strip()[:100]}...")
    except:
        pass
    
    if i % 30 == 0:  # Progress update every 30 seconds
        print(f"⏱️ {i//60}:{i%60:02d} - Server starting...")
    
    time.sleep(1)
else:
    print("\n✅ Server startup completed! Ready for testing.")

# Show last few lines of output
if startup_lines:
    print("\n📋 Recent server output:")
    for line in startup_lines[-5:]:
        print(f"  {line}")

## Step 5: Test Server Health

Check if the server is responding properly:

In [ ]:
import requests
import time
from IPython.display import HTML, display

print("🔍 Testing Server Health...")

# Test server health endpoint
max_retries = 36  # 6 minutes
server_ready = False

for attempt in range(max_retries):
    try:
        # Test health endpoint
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            print("✅ Server is ready and healthy!")
            server_ready = True
            break
        else:
            print(f"⚠️ Health check returned: {response.status_code}")
            
    except requests.exceptions.ConnectionError:
        print(f"⏳ Waiting for server... ({attempt+1}/{max_retries})")
    except requests.exceptions.Timeout:
        print(f"⏰ Server timeout... ({attempt+1}/{max_retries})")
    except Exception as e:
        print(f"❌ Error: {e}")
    
    time.sleep(10)

# Check if server process is still running
if server_process.poll() is not None:
    print(f"❌ Server process died with code: {server_process.returncode}")
    print("\n🔍 Last server output:")
    try:
        # Read any remaining output
        remaining_output = server_process.stdout.read()
        if remaining_output:
            lines = remaining_output.split('\n')[-10:]  # Last 10 lines
            for line in lines:
                if line.strip():
                    print(f"  {line}")
    except:
        pass
elif not server_ready:
    print("⚠️ Server is running but not responding to health checks")
    print("💡 Try proceeding with TTS test anyway")

if server_ready:
    # Test models endpoint
    try:
        models_response = requests.get("http://localhost:8000/v1/models", timeout=5)
        if models_response.status_code == 200:
            models = models_response.json()
            print(f"✅ Available models: {[m['id'] for m in models.get('data', [])]}")
    except:
        print("⚠️ Could not list models")

print("\n🎯 Server test completed!")

## Step 6: Generate Speech

Test the TTS functionality:

In [ ]:
import requests
import io
import soundfile as sf
from IPython.display import Audio, display
import time

print("🎵 Testing Voxtral Text-to-Speech...")

# Test text (keep it short for T4)
test_text = "Hello! This is Voxtral TTS on T4 GPU."

# TTS request payload
payload = {
    "input": test_text,
    "model": "mistralai/Voxtral-4B-TTS-2603",
    "response_format": "wav",
    "voice": "casual_male",
    "speed": 1.0
}

print(f"📝 Text: '{test_text}'")
print(f"🎤 Voice: casual_male")
print("⏳ Generating speech...")

try:
    start_time = time.time()
    
    response = requests.post(
        "http://localhost:8000/v1/audio/speech",
        json=payload,
        timeout=180.0  # 3 minute timeout
    )
    
    generation_time = time.time() - start_time
    
    if response.status_code == 200:
        # Load and play the audio
        audio_array, sr = sf.read(io.BytesIO(response.content), dtype="float32")
        duration = len(audio_array) / sr
        
        print(f"\n✅ Speech generated successfully!")
        print(f"⏱️ Generation time: {generation_time:.2f} seconds")
        print(f"🎵 Audio: {len(audio_array)} samples at {sr} Hz")
        print(f"⏱️ Duration: {duration:.2f} seconds")
        print(f"⚡ Real-time factor: {generation_time/duration:.2f}x")
        
        # Display audio player
        print("\n🎧 Click to play:")
        display(Audio(data=audio_array, rate=sr))
        
    else:
        print(f"❌ TTS Error: {response.status_code}")
        print(f"Response: {response.text[:500]}")
        
except requests.exceptions.Timeout:
    print("⏰ TTS generation timed out (try shorter text)")
except requests.exceptions.ConnectionError:
    print("🔌 Server connection error - server may not be ready")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

## Step 7: Test Different Voices

Try different preset voices:

In [ ]:
# Test different voices
voice_tests = [
    {"text": "Bonjour, je suis Voxtral.", "voice": "casual_female", "lang": "French"},
    {"text": "Hola, soy Voxtral.", "voice": "professional_male", "lang": "Spanish"},
    {"text": "Hallo, ich bin Voxtral.", "voice": "casual_male", "lang": "German"}
]

for i, test in enumerate(voice_tests):
    print(f"\n🎤 Testing {test['lang']} ({test['voice']})...")
    
    payload = {
        "input": test["text"],
        "model": "mistralai/Voxtral-4B-TTS-2603",
        "response_format": "wav",
        "voice": test["voice"]
    }
    
    try:
        response = requests.post(
            "http://localhost:8000/v1/audio/speech",
            json=payload,
            timeout=60.0
        )
        
        if response.status_code == 200:
            audio_array, sr = sf.read(io.BytesIO(response.content), dtype="float32")
            print(f"✅ {test['lang']} audio: {len(audio_array)/sr:.2f}s")
            display(Audio(data=audio_array, rate=sr))
        else:
            print(f"❌ {test['lang']} failed: {response.status_code}")
            
    except Exception as e:
        print(f"❌ {test['lang']} error: {e}")
    
    time.sleep(2)  # Brief pause between tests

## 🔧 Troubleshooting Guide

### Common T4 Issues & Solutions:

#### ❌ cuFFT Factory Error
**Solution**: Runtime → Restart runtime → Run all cells again

#### ❌ Out of Memory
**Solutions**:
- Reduce `--max-model-len` to 64 or 32
- Reduce `--gpu-memory-utilization` to 0.25
- Restart runtime completely

#### ❌ Server Not Responding
**Solutions**:
- Wait longer (T4 is slower)
- Check if server process is still running
- Try without `--omni` flag first

#### ❌ BFloat16 Error
**Solutions**:
- Ensure Step 3 completed successfully
- Check that BFloat16 patterns were found and fixed
- Restart runtime if needed

### Performance Tips for T4:
- Use short text (under 100 characters)
- Be patient (T4 is slower than A100)
- Monitor GPU memory usage
- Restart runtime if performance degrades

### Available Voices:
- `casual_male`, `casual_female`
- `professional_male`, `professional_female`
- And 16+ more preset voices

## 🧹 Cleanup

Stop the server when finished:

In [ ]:
# Clean up server and GPU memory
try:
    if 'server_process' in locals():
        # Kill the entire process group
        os.killpg(os.getpgid(server_process.pid), signal.SIGTERM)
        server_process.wait(timeout=10)
        print("✅ Server stopped successfully")
except:
    try:
        # Fallback: kill by name
        subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
        print("✅ Server killed via process name")
    except:
        print("⚠️ Could not stop server cleanly")

# Clear GPU memory
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    print("✅ GPU memory cleared")

print("\n🎉 Voxtral TTS session completed!")
print("💾 Save your work and enjoy the generated audio!")